# Getting Started with NPAP

**NPAP** (Network Partitioning and Aggregation Package) is a standalone, pip-installable
open-source Python library for **reducing the spatial complexity of network graphs**.
Built on top of [NetworkX](https://networkx.org/), it is decoupled from any specific
modelling framework and can therefore be readily integrated into existing tools or used
in isolation.

A key design decision in NPAP is the **explicit separation of the network reduction
process into two distinct steps**:

1. **Partitioning** — maps each vertex (node) in the original network to a group
   (cluster). This step only determines group membership; it does not modify the graph
   itself.
2. **Aggregation** — reduces the network topology based on a given partitioning result by
   aggregating nodes, edges, and their associated properties.

This separation gives users fine-grained control: partitioning algorithms can be swapped
independently of the aggregation method, different aggregation strategies can be applied
to the same partitioning result, or either step can be used in isolation. NPAP follows a
**strategy pattern** architecture so that users can register custom data loaders,
partitioning algorithms, and aggregation rules without modifying the core code. The
library currently provides **13 partitioning strategies** organised in four families and
**two pre-defined aggregation profiles**.

Although NPAP was initially developed with a focus on power systems, its architecture is
general-purpose and applicable to any network graph. The target audience therefore
includes power systems researchers, energy modelers, network analysts, and more broadly,
anyone interested in reducing the spatial complexity of a graph.

This notebook is a **10-minute tour** that shows you how to:

1. Install the library and initialise the `PartitionAggregatorManager` facade (main manager).
2. Load data into NPAP.
3. Pre-process the graph (e.g., aggregate parallel edges).
4. Partition the network into clusters.
5. Aggregate each cluster into a representative node via an `AggregationProfile`.
6. Visualise the network throughout the pipeline.

Section 2 runs the full pipeline on a **general-purpose** focus. Section 3
closes with a short **power-systems** example illustrating NPAP's voltage-aware and
AC-island-aware partitioning on real European high-voltage grid data.

📚 Full documentation: **<https://npap.readthedocs.io>**

## Pipeline Architecture

The full NPAP pipeline is shown below. The whole workflow is orchestrated and accessed
through a single facade (main manager) — the `PartitionAggregatorManager` — which coordinates three
specialised manager classes (`InputDataManager`, `PartitioningManager`, and
`AggregationManager`). Each stage of the pipeline is backed by a pluggable strategy and
can be illustrated at any point through the visualization component.

![NPAP pipeline architecture](figures/pipeline-architecture.svg)

NPAP first prepares the network graph in two stages: data is loaded and a NetworkX graph
is created and validated, then optional pre-processing steps (e.g., parallel-edge
aggregation, voltage-level grouping) prepare the graph for the reduction process. The
reduction itself is then split into the partitioning and aggregation steps introduced
above.

See the [Available Strategies guide](https://npap.readthedocs.io/en/latest/user-guide/available-strategies.html)
for the complete catalogue of loaders, partitioners and aggregators.

## 1. Installation

NPAP is released under the MIT license and published on PyPI as a standalone,
pip-installable package. Uncomment the cell below if you do not have it yet.

See the [installation guide](https://npap.readthedocs.io/en/latest/user-guide/installation.html) for details.

In [37]:
# %pip install npap

## 2. General-Purpose Workflow — A Real Geo-Referenced Network

To highlight that NPAP's core workflow is **domain-agnostic**, this section runs the
full pipeline on the **European high-voltage grid** (~6,800 buses, 35 countries) derived
from OpenStreetMap data and hosted on
[Zenodo](https://doi.org/10.5281/zenodo.18619025) by Xiong et al. (2025). At this stage
we deliberately **ignore the electrical semantics** of the dataset and treat it as a
generic geo-referenced graph: vertices have coordinates, edges have a length and a
capacity. Section 3 then re-loads the *same* network with NPAP's power-systems features
enabled, so you can compare the two treatments side-by-side.

### 2.1 Initialise the Facade Manager

The whole NPAP workflow is orchestrated and accessed through the
[`PartitionAggregatorManager`](https://npap.readthedocs.io/en/latest/api/npap.html#npap.PartitionAggregatorManager)
facade, which internally coordinates the data-loading, partitioning, and aggregation
managers.

In [38]:
import networkx as nx
import pandas as pd
from utils import download_and_preprocess_zenodo_data

import npap
from npap import AggregationProfile

manager = npap.PartitionAggregatorManager()

data_files = download_and_preprocess_zenodo_data()

Fetching PyPSA-Eur network data from Zenodo:
  buses.csv: already cached (0.8 MB)
  lines.csv: already cached (1.5 MB)
  transformers.csv: already cached (0.1 MB)
  converters.csv: already cached (0.0 MB)
  links.csv: already cached (0.3 MB)


### 2.2 Data Loading

NPAP graphs can be supplied in three ways: directly as a NetworkX graph
(`networkx_direct`), through a generic node/edge CSV pair (`csv_files`), or through the
power-systems–oriented `va_loader` (buses, lines, transformers, converters, DC links).
Custom data-loading strategies can be registered seamlessly thanks to the strategy
pattern.

For a generic treatment of the network we use **`csv_files`**: it only needs a node file
and an edge file, and treats every edge uniformly with no type differentiation. This is
the right choice whenever the full physical topology is not required.

See the [data-loading guide](https://npap.readthedocs.io/en/latest/user-guide/data-loading.html) for the full list of strategies.

In [39]:
graph = manager.load_data(
    strategy="csv_files",
    node_file=str(data_files["buses.csv"]),
    edge_file=str(data_files["lines.csv"]),
)

print(f"Graph type : {type(graph).__name__}")
print(f"Vertices   : {graph.number_of_nodes():,}")
print(f"Edges      : {graph.number_of_edges():,}")

Graph type : MultiDiGraph
Vertices   : 6,863
Edges      : 9,162


C:\Users\Marco\Documents\git\NPAP\npap\input\csv_loader.py:197: UserWarning:

Parallel edges detected in CSV edge file. A MultiDiGraph will be created. Call manager.aggregate_parallel_edges() to collapse parallel edges before partitioning.



In [ ]:
manager.plot_network(style="simple", title="European HV Grid — Raw");

### 2.3 Pre-processing: Aggregate Parallel Edges

Partitioning strategies in NPAP operate on simple `DiGraph`s. The PyPSA-Eur dataset
contains many double-circuit lines — two edges sharing the same pair of vertices — so
the `csv_files` loader returns a `MultiDiGraph` and emits a warning. We collapse the
parallel edges through the facade manager and explicitly declare how each edge property
should be combined.

In [ ]:
if isinstance(graph, nx.MultiDiGraph):
    edges_before = graph.number_of_edges()
    graph = manager.aggregate_parallel_edges(
        edge_properties={
            "length": "average",  # mean edge length
            "s_nom": "sum",  # total capacity of parallel edges
        },
        default_strategy="average",
        warn_on_defaults=False,
    )
    print(f"Edges before: {edges_before:,}")
    print(f"Edges after : {graph.number_of_edges():,}")

### 2.4 Partitioning

The partitioning step assigns each vertex of the network to a cluster — without yet
modifying the graph itself. NPAP currently provides **four families of partitioning
strategies** combining geographical and electrical node distance with the option of
treating voltage levels independently (voltage-awareness). Each family supports up to
13 algorithms, including **k-means**, **k-medoids**, **DBSCAN**, **HDBSCAN**, and
**hierarchical clustering**.

For a generic geo-referenced graph like ours, geographical distance (latitudes and
longitudes) is the natural choice. We use **k-medoids with the Haversine metric** to
obtain spatially compact clusters. The outcome is returned as a `PartitionResult` that
stores the mapping together with metadata such as the number of clusters and the strategy
name.

See the [partitioning guide](https://npap.readthedocs.io/en/latest/user-guide/partitioning/index.html) for details on each strategy.

In [ ]:
N_CLUSTERS = 200

partition = manager.partition(
    strategy="geographical_kmedoids_haversine",
    n_clusters=N_CLUSTERS,
)

sizes = pd.Series({k: len(v) for k, v in partition.mapping.items()})
print(f"Clusters created : {partition.n_clusters}")
print(f"Cluster size — min: {sizes.min()}, max: {sizes.max()}, mean: {sizes.mean():.1f}")

manager.plot_network(style="clustered", title=f"Partitioned ({N_CLUSTERS} clusters)");

### 2.5 Aggregation with an `AggregationProfile`

The aggregation step reduces the network topology based on the partitioning result. NPAP
decomposes this process into a **three-tier pipeline**:

1. **Topology tier** — builds a new graph by creating one representative vertex per
   cluster and adjusting the edges between them accordingly.
2. **Domain-specific tier** *(optional)* — modifies physical properties of the resulting
   graph (e.g., adding edges that did not exist in the original graph or overriding
   specific quantities such as line reactances).
3. **Property aggregation tier** — applies user-specified functions (`sum`, `average`,
   `equivalent_reactance`, …) to the remaining node and edge properties.

These tiers are configured through an
[`AggregationProfile`](https://npap.readthedocs.io/en/latest/user-guide/aggregation.html).
Below we average vertex coordinates, take the first `voltage` per cluster, sum the edge
capacities, and average the edge lengths.

In [ ]:
profile = AggregationProfile(
    topology_strategy="simple",
    node_properties={
        "lat": "average",
        "lon": "average",
        "voltage": "first",
    },
    edge_properties={
        "s_nom": "sum",
        "length": "average",
    },
    default_node_strategy="average",
    default_edge_strategy="sum",
    warn_on_defaults=False,
)

nodes_before = manager.get_current_graph().number_of_nodes()
edges_before = manager.get_current_graph().number_of_edges()

agg_graph = manager.aggregate(profile=profile)

print(
    f"Nodes: {nodes_before:,} -> {agg_graph.number_of_nodes():,} "
    f"({agg_graph.number_of_nodes() / nodes_before:.1%} of original)"
)
print(
    f"Edges: {edges_before:,} -> {agg_graph.number_of_edges():,} "
    f"({agg_graph.number_of_edges() / edges_before:.1%} of original)"
)

### 2.6 Visualise the Aggregated Network

The network can be illustrated throughout the pipeline using NPAP's built-in
visualization component, powered by Plotly. Available styles are `simple`, `clustered`,
and `voltage_aware`. See the [visualization guide](https://npap.readthedocs.io/en/latest/user-guide/visualization.html) for more.

In [ ]:
manager.plot_network(style="simple", title=f"Aggregated ({N_CLUSTERS} nodes)");

## 3. Power-Systems Workflow — Voltage-Aware Partitioning on the Same Network

We now re-load the **same European HV grid** with NPAP's power-systems features enabled,
so the contrast with Section 2 is immediate. The `va_loader` ingests all five PyPSA-Eur
files (buses, lines, transformers, converters, DC links) and builds a physically detailed
graph in which every edge is typed (`line` / `trafo` / `dc_link`) and every bus is
assigned to an **AC island** — the set of buses reachable without crossing a DC link.

NPAP automatically detects AC islands linked solely by DC interconnections and partitions
them independently. AC-island and voltage-level constraints are enforced by setting the
distance-matrix entries between nodes in different islands or voltage levels to infinity,
which makes both mechanisms **algorithm-agnostic**: they work with any distance-based
partitioning method without modifying the algorithm itself.

|                          | Section 2 (simple)                  | Section 3 (voltage-aware)              |
|--------------------------|-------------------------------------|----------------------------------------|
| Data loader              | `csv_files`                         | `va_loader`                            |
| Edge types               | all uniform                         | line / trafo / DC link                 |
| AC islands               | ✗                                   | ✓                                      |
| Voltage-level awareness  | ✗                                   | ✓                                      |
| Partitioning strategy    | `geographical_kmedoids_haversine`   | `va_geographical_kmedoids_haversine`   |
| Per-edge-type aggregation | ✗                                  | ✓                                      |

In [ ]:
va_manager = npap.PartitionAggregatorManager()

va_graph = va_manager.load_data(
    strategy="va_loader",
    node_file=str(data_files["buses.csv"]),
    line_file=str(data_files["lines.csv"]),
    transformer_file=str(data_files["transformers.csv"]),
    converter_file=str(data_files["converters.csv"]),
    link_file=str(data_files["links.csv"]),
)

print(f"Nodes: {va_graph.number_of_nodes():,}  |  Edges: {va_graph.number_of_edges():,}")

In [ ]:
# Collapse parallel double-circuit lines and harmonise voltage levels.
if isinstance(va_graph, nx.MultiDiGraph):
    va_graph = va_manager.aggregate_parallel_edges(
        edge_properties={
            "x": "average",
            "r": "average",
            "s_nom": "sum",
            "length": "average",
        },
        default_strategy="first",
        warn_on_defaults=False,
    )

va_manager.group_by_voltage_levels([220, 380])
va_manager.plot_network(style="voltage_aware", title="European HV Grid — VA Load");

In [ ]:
va_partition = va_manager.partition(
    strategy="va_geographical_kmedoids_haversine",
    n_clusters=N_CLUSTERS,
)

va_manager.plot_network(style="clustered", title=f"Voltage-Aware — {N_CLUSTERS} clusters");

In [ ]:
# Per-edge-type aggregation: different physical laws apply to lines, trafos and DC links.
va_profile = AggregationProfile(
    topology_strategy="simple",
    node_properties={
        "lat": "average",
        "lon": "average",
        "voltage": "first",
        "country": "first",
    },
    edge_type_properties={
        "line": {
            "x": "equivalent_reactance",
            "r": "equivalent_reactance",
            "s_nom": "sum",
            "length": "average",
        },
        "trafo": {"x": "equivalent_reactance", "s_nom": "sum"},
        "dc_link": {"p_nom": "sum", "length": "average"},
    },
    default_node_strategy="average",
    default_edge_strategy="sum",
    warn_on_defaults=False,
)

va_agg = va_manager.aggregate(profile=va_profile)
va_manager.plot_network(style="voltage_aware", title="Voltage-Aware — Aggregated");

## 4. Wrap-up

In ten minutes you have walked through the full NPAP pipeline for spatial complexity
reduction:

1. **Install** — `pip install npap` (standalone, framework-agnostic, MIT-licensed).
2. **Initialise** — `manager = npap.PartitionAggregatorManager()` (single facade entry
   point).
3. **Load** — choose a data-loading strategy (`csv_files`, `networkx_direct`,
   `va_loader`) or register your own.
4. **Pre-process** — optional steps such as `aggregate_parallel_edges` or, for power
   grids, `group_by_voltage_levels`.
5. **Partitioning** — 13 strategies across four families (geographical / electrical, each
   with optional voltage-awareness); produces a `PartitionResult`.
6. **Aggregation** — three-tier pipeline (topology → optional domain-specific →
   property aggregation), fully configurable through an `AggregationProfile`.
7. **Visualise** — interactive Plotly maps (`simple`, `clustered`, `voltage_aware`)
   available at every stage of the pipeline.

### Where to go next

- [Available strategies](https://npap.readthedocs.io/en/latest/user-guide/available-strategies.html) — the full catalogue of loaders, partitioners and aggregators.
- [Extending NPAP](https://npap.readthedocs.io/en/latest/user-guide/extending.html) — register your own data loader, partitioning or aggregation strategy via the strategy pattern.
- [API reference](https://npap.readthedocs.io/en/latest/api/npap.html).
- [GitHub](https://github.com/IEE-TUGraz/NPAP) — source code, issues, contributions.